In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
products = pd.read_csv("../data/raw/olist_products_dataset.csv")

In [3]:
# First rows
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [11]:

# 1. Inspect the dataset
print("Original shape:", products.shape)
print("\nMissing values:")
print(products.isnull().sum())

# 2. Check duplicate product IDs
print("\nDuplicate product IDs:", products["product_id"].duplicated().sum())

# 3. Check fully duplicated rows
print("Fully duplicated rows:", products.duplicated().sum())

# 4. Check numerical values
numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

print("\nNumerical summary:")
print(products[numeric_cols].describe())

# 5. Check missing product attributes
attribute_cols = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

print("\nMissing values in product attributes:")
print(products[attribute_cols].isnull().sum())


Original shape: (32951, 9)

Missing values:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Duplicate product IDs: 0
Fully duplicated rows: 0

Numerical summary:
       product_name_lenght  product_description_lenght  product_photos_qty  \
count         32341.000000                32341.000000        32341.000000   
mean             48.476949                  771.495285            2.188986   
std              10.245741                  635.115225            1.736766   
min               5.000000                    4.000000            1.000000   
25%              42.000000                  339.000000            1.000000   
50%              51.000000                  595.000000            1.000000   
75%              57.

In [12]:

# 1. Check whether the 610 missing attribute values occur in the same rows
attribute_cols = [
    "product_category_name",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty"
]

print("Rows missing all four attributes:",
      products[attribute_cols].isnull().all(axis=1).sum())

print("\nRows missing some but not all attributes:",
      products[attribute_cols].isnull().any(axis=1).sum()
      - products[attribute_cols].isnull().all(axis=1).sum())

# 2. Inspect rows with missing physical measurements
physical_cols = [
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

missing_physical = products[products[physical_cols].isnull().any(axis=1)]

print("\nProducts with missing physical measurements:")
print(missing_physical.to_string(index=False))

# 3. Investigate products with zero weight
zero_weight = products[products["product_weight_g"] == 0]

print("\nNumber of products with zero weight:", len(zero_weight))
print("\nProducts with zero weight:")
print(zero_weight.to_string(index=False))

# 4. Check for impossible negative values
numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

print("\nNegative values by column:")
print((products[numeric_cols] < 0).sum())


Rows missing all four attributes: 610

Rows missing some but not all attributes: 0

Products with missing physical measurements:
                      product_id product_category_name  product_name_lenght  product_description_lenght  product_photos_qty  product_weight_g  product_length_cm  product_height_cm  product_width_cm
09ff539a621711667c43eba6a3bd8466                 bebes                 60.0                       865.0                 3.0               NaN                NaN                NaN               NaN
5eb564652db742ff8f28759cd8d2652a                   NaN                  NaN                         NaN                 NaN               NaN                NaN                NaN               NaN

Number of products with zero weight: 4

Products with zero weight:
                      product_id product_category_name  product_name_lenght  product_description_lenght  product_photos_qty  product_weight_g  product_length_cm  product_height_cm  product_width_cm
81781c0fed9

In [13]:

import pandas as pd

# Create a separate copy for cleaning
products_cleaned = products.copy()

# 1. Correct spelling in column names
products_cleaned = products_cleaned.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
})

# 2. Standardize missing product categories
products_cleaned["product_category_name"] = (
    products_cleaned["product_category_name"].fillna("unknown")
)

# 3. Treat zero product weights as missing values
products_cleaned.loc[
    products_cleaned["product_weight_g"] == 0,
    "product_weight_g"
] = float("nan")

# 4. Check the results
print("Original dataset shape:", products.shape)
print("Cleaned working dataset shape:", products_cleaned.shape)

print("\nMissing values after cleaning:")
print(products_cleaned.isnull().sum())

print("\nZero weights remaining:",
      (products_cleaned["product_weight_g"] == 0).sum())

print("\nDuplicate product IDs:",
      products_cleaned["product_id"].duplicated().sum())

print("\nUpdated columns:")
print(products_cleaned.columns.tolist())


Original dataset shape: (32951, 9)
Cleaned working dataset shape: (32951, 9)

Missing values after cleaning:
product_id                      0
product_category_name           0
product_name_length           610
product_description_length    610
product_photos_qty            610
product_weight_g                6
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Zero weights remaining: 0

Duplicate product IDs: 0

Updated columns:
['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']


In [14]:

# Final validation of products_cleaned

print("Dataset shape:", products_cleaned.shape)

print("\nColumn names:")
print(products_cleaned.columns.tolist())

print("\nDuplicate product IDs:",
      products_cleaned["product_id"].duplicated().sum())

print("\nMissing categories:",
      products_cleaned["product_category_name"].isnull().sum())

print("\nUnknown categories:",
      (products_cleaned["product_category_name"] == "unknown").sum())

print("\nMissing values:")
print(products_cleaned.isnull().sum())

print("\nZero weights remaining:",
      (products_cleaned["product_weight_g"] == 0).sum())

print("\nNegative numerical values:")
numeric_cols = products_cleaned.select_dtypes(include="number").columns
print((products_cleaned[numeric_cols] < 0).sum())

print("\nPreview of cleaned dataset:")
print(products_cleaned.head())


Dataset shape: (32951, 9)

Column names:
['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

Duplicate product IDs: 0

Missing categories: 0

Unknown categories: 610

Missing values:
product_id                      0
product_category_name           0
product_name_length           610
product_description_length    610
product_photos_qty            610
product_weight_g                6
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Zero weights remaining: 0

Negative numerical values:
product_name_length           0
product_description_length    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64

Preview of cleaned dataset:
                         produc

In [15]:

products_cleaned.to_csv("products_cleaned.csv", index=False)

print("Cleaned dataset saved as products_cleaned.csv")


Cleaned dataset saved as products_cleaned.csv


Products Dataset – Data Cleaning Summary

The Olist products dataset originally contained 32,951 rows and 9 columns.

Cleaning Steps

Checked for duplicate product IDs and fully duplicated rows; none were found.
Corrected the spelling of product_name_lenght to product_name_length and product_description_lenght to product_description_length.
Replaced missing product categories with "unknown" (610 products).
Replaced zero product weights with missing values (4 products).
Preserved other missing values without deleting product records.
Checked numerical columns for negative values; none were found.
Final Result

The cleaned dataset retains all 32,951 rows and 9 columns. The original dataset remains unchanged, and the cleaned version is saved as products_cleaned.csv.

Relevance to Research

Product categories will support the analysis of customer satisfaction across categories (Research Question 4). This dataset can also be joined with Olist order, order-item, and review datasets for further analysis of delivery performance and customer satisfaction.